In [16]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np

In [2]:
dwell_loc = pd.read_csv("../data/loc_code_detection_patterns.csv")

In [3]:
direction = dwell_loc[['tag_id', 'loc_code', 'start_time', 'end_time', 'number_of_detections', 'dwell_time']]
direction = direction[direction['loc_code']!='3M'].replace('60','61') # remove 3M, 60 and 61 are the same-so replace
direction

,tag_id,loc_code,start_time,end_time,number_of_detections,dwell_time
0,989.001026,202,2018-05-30 12:44:10.620000,2018-05-30 12:44:10.620000,1,0 days 00:00:00
1,989.001026,202,2018-05-30 13:11:46.610000,2018-05-30 13:11:46.610000,1,0 days 00:00:00
2,989.001026,201,2018-05-30 15:10:44.680000,2018-05-30 15:10:46.230000,2,0 days 00:00:01.550000
3,989.001026,202,2018-05-30 15:11:52,2018-05-30 15:11:52,1,0 days 00:00:00
4,989.001026,202,2018-05-30 21:19:02.240000,2018-05-30 21:19:02.240000,1,0 days 00:00:00
...,...,...,...,...,...,...
231272,989.002028,51,2025-03-09 13:39:35.920000,2025-03-09 13:40:40.040000,2,0 days 00:01:04.120000
231273,989.002028,302,2025-03-10 06:39:50.720000,2025-03-10 06:39:50.720000,1,0 days 00:00:00
231274,989.002027,6A,2025-03-12 00:58:17.600000,2025-03-12 00:58:17.600000,1,0 days 00:00:00
231275,989.002027,6B,2025-03-12 01:00:24.080000,2025-03-12 01:00:24.080000,1,0 days 00:00:00


In [4]:
direction["start_time"] = pd.to_datetime(direction["start_time"], errors="coerce")
direction["end_time"] = pd.to_datetime(direction["end_time"], errors="coerce")

def calculate_sequences(group):
    
    group["start_shift"] = group["start_time"].shift(-1)
    group["time_between_detection"] = group["start_shift"] - group["end_time"]
    return {
        
        "loc_code": group["loc_code"].tolist(),
        "time_between_detections": group["time_between_detection"].tolist(),
        "number_of_detections": group["number_of_detections"].tolist(),
        "dwell_time": group["dwell_time"].tolist()
    }

sequence_data = direction.groupby("tag_id").apply(calculate_sequences).reset_index(name="sequence_data")

print(sequence_data)

            tag_id                                      sequence_data
0         0.000061  {'loc_code': ['701'], 'time_between_detections...
1         0.000081  {'loc_code': ['701'], 'time_between_detections...
2         0.000174  {'loc_code': ['21'], 'time_between_detections'...
3         0.000183  {'loc_code': ['702'], 'time_between_detections...
4         3.237301  {'loc_code': ['11'], 'time_between_detections'...
...            ...                                                ...
109059  989.002028  {'loc_code': ['C1'], 'time_between_detections'...
109060  989.002028  {'loc_code': ['C2', 'C1'], 'time_between_detec...
109061  989.005135  {'loc_code': ['9C'], 'time_between_detections'...
109062  990.000004  {'loc_code': ['501'], 'time_between_detections...
109063  997.131499  {'loc_code': ['6B'], 'time_between_detections'...

[109064 rows x 2 columns]


In [8]:
sequence_df = pd.DataFrame(sequence_data)
sequence_df["loc_code"] = sequence_df['sequence_data'].apply(lambda x: x["loc_code"])
sequence_df["time_between_locations"] = sequence_df['sequence_data'].apply(lambda x: x["time_between_detections"])
sequence_df["number_of_detections"] = sequence_df['sequence_data'].apply(lambda x: x["number_of_detections"])
sequence_df["dwell_time"] = sequence_df['sequence_data'].apply(lambda x: x["dwell_time"])
sequence_df = sequence_df.drop(columns=['sequence_data'])

sequence_df

,tag_id,loc_code,time_between_locations,number_of_detections,dwell_time
0,0.000061,[701],[NaT],[1],[0 days 00:00:00]
1,0.000081,[701],[NaT],[1],[0 days 00:00:00]
2,0.000174,[21],[NaT],[1],[0 days 00:00:00]
3,0.000183,[702],[NaT],[1],[0 days 00:00:00]
4,3.237301,[11],[NaT],[1],[0 days 00:00:00]
...,...,...,...,...,...
109059,989.002028,[C1],[NaT],[4],[0 days 00:03:00.020000]
109060,989.002028,"[C2, C1]","[0 days 00:54:12.480000, NaT]","[4, 1]","[0 days 00:02:59.920000, 0 days 00:00:00]"
109061,989.005135,[9C],[NaT],[2],[0 days 00:00:00]
109062,990.000004,[501],[NaT],[4],[430 days 23:41:50.140000]


In [10]:
# time_between_locations' > 30 days
possible_outmigrants = sequence_df[sequence_df["time_between_locations"].apply(lambda x: any(t > pd.Timedelta(days=30) for t in x))]

possible_outmigrants

,tag_id,loc_code,time_between_locations,number_of_detections,dwell_time
32,956.000005,"[6A, 61, 61, 6A, 61]","[21 days 16:26:32.950000, 698 days 02:14:13.63...","[5, 6, 12, 1, 548]","[2 days 10:58:30.360000, 1 days 20:15:12.63000..."
34,956.000006,"[6A, 501]","[217 days 18:46:49.710000, NaT]","[11, 116]","[0 days 02:55:09.300000, 886 days 23:37:28.230..."
178,982.000412,"[83, 84]","[171 days 07:33:44.040000, NaT]","[5, 5]","[0 days 00:50:44.900000, 0 days 00:27:02.150000]"
199,982.000412,"[83, 84]","[120 days 01:29:17.290000, NaT]","[3, 1]","[0 days 00:19:01.560000, 0 days 00:00:00]"
203,982.000412,"[83, 84]","[165 days 20:50:11.020000, NaT]","[332, 26]","[0 days 18:00:58.240000, 0 days 21:40:09.280000]"
...,...,...,...,...,...
108395,989.002028,"[402, 45, 44, 45, 44, 45, 44, 45, 44, 45, 44, ...","[273 days 08:23:33.220000, 0 days 00:01:08, 0 ...","[3, 3, 2, 1, 1, 2, 1, 1, 1, 98, 2, 1, 1, 1, 1]","[0 days 00:06:22.220000, 0 days 04:55:48.07000..."
108438,989.002028,"[402, 401, 402]","[282 days 00:37:17.730000, 0 days 05:17:43.940...","[1, 1, 124]","[0 days 00:00:00, 0 days 00:00:00, 0 days 04:3..."
108839,989.002028,"[402, 401, 402, 401, 402, 401, 402, 401, 402, ...","[1 days 00:26:21.140000, 0 days 21:27:48.27000...","[4, 7, 1, 1, 3, 73, 3, 2, 23, 1, 41, 1, 23, 1,...","[0 days 22:12:32.860000, 0 days 02:41:05.42000..."
108889,989.002028,"[44, 45, 44]","[0 days 00:02:12.890000, 117 days 21:18:35.350...","[6, 2, 1]","[0 days 00:03:59.710000, 0 days 00:00:00, 0 da..."


There are couple tags that follow the same route, eg; 34 tags that moved in the exact combination ['7C', '701', '7A', '7B']

In [24]:
all_items = [item for sublist in possible_outmigrants['loc_code'] for item in [sublist]]
counts = pd.Series(all_items).value_counts()
movement_counts = pd.DataFrame({'Combination': counts.index, 'Count': counts.values})
movement_counts

,Combination,Count
0,"[7C, 701, 7A, 7B]",34
1,"[82, 83, 84]",26
2,"[83, 84]",26
3,"[12, 11, 12]",25
4,"[7C, 7A, 7B]",25
...,...,...
769,"[302, 301, 302, 301, 302, 301, 302, 301]",1
770,"[71, 701, 702, 401, 402, A1, 21, 922, 921, 61,...",1
771,"[A1, A2, C2, C1, C2, C1, C2, C1, C2]",1
772,"[82, 81, 82, 81, 83, 84]",1
